# Train the Pokémon TCG Deep Sets evaluator

Play fresh games with Kaggle’s native CABT engine, fit a position evaluator, then let a frozen copy play both seats to collect the next training set. The last cell checks the saved weights against real observations and legal selections. A CPU runtime is sufficient for the defaults.

**Colab: choose Runtime → Run all.** Setup clones the public [project repository](https://github.com/le21-j/pokemon-tcg-fpga) and installs dependencies automatically.

In [ ]:
# @title 1. Automatic setup {display-mode: "form"}
# @markdown Run All clones the public project and installs dependencies automatically in Colab. Optional overrides can stay at their defaults.
REPOSITORY_PATH = "" # @param {type:"string"}
MOUNT_GOOGLE_DRIVE = False # @param {type:"boolean"}
INSTALL_LOCALLY = False # @param {type:"boolean"}

"""Standard-library Colab setup for the public Pokémon TCG project repository."""

import os
from pathlib import Path
import subprocess
import sys
import tempfile


PUBLIC_REPOSITORY = "https://github.com/le21-j/pokemon-tcg-fpga.git"


def running_in_colab():
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        return True
    try:
        import google.colab
        return True
    except ImportError:
        return False


def _project_ready(root):
    return all((root / name).is_file() for name in (
        "project/arena.py", "project/sample_sdk.py", "project/agent/model.py",
        "project/rl_baseline/train.py", "project/agent_template/agent.py",
        "project/requirements-colab.txt", "project/train_deep_sets.py",
        "project/check_deep_sets_training.py"))


def setup_project(repository_path="", *, repository_url=PUBLIC_REPOSITORY, repository_ref="main",
                  in_colab=None, workspace_parent=None, install_locally=False):
    in_colab = running_in_colab() if in_colab is None else in_colab
    if repository_path.strip():
        root = Path(repository_path).expanduser().resolve()
        if not _project_ready(root):
            raise ValueError("REPOSITORY_PATH must contain the Pokémon TCG project")
    elif not in_colab:
        root = next((p for p in [Path.cwd(), *Path.cwd().parents] if _project_ready(p)), None)
        if root is None:
            raise ValueError("Run inside the project checkout or set REPOSITORY_PATH")
    else:
        parent = Path(workspace_parent) if workspace_parent is not None else (
            Path("/content") if Path("/content").is_dir() else Path.cwd())
        parent.mkdir(parents=True, exist_ok=True)
        name = "pokemon-tcg-fpga"
        root = parent / name
        retry = 0
        while root.exists() and not _project_ready(root):
            retry += 1
            root = parent / f"{name}-{retry}"
        if not root.exists():
            with tempfile.TemporaryDirectory(prefix="tcg-clone-", dir=parent) as temporary:
                checkout = Path(temporary) / "checkout"
                subprocess.run(["git", "clone", "--depth", "1", "--branch", repository_ref,
                                repository_url, str(checkout)],
                               env=dict(os.environ, GIT_TERMINAL_PROMPT="0"), check=True)
                if not _project_ready(checkout):
                    raise RuntimeError("The downloaded repository is missing project files")
                checkout.rename(root)
            print("Cloned the public Pokémon TCG project")
    if in_colab or install_locally:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
                               "-r", str(root / "project/requirements-colab.txt")])
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print("Project ready:", root)
    return root


if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = setup_project(REPOSITORY_PATH, install_locally=INSTALL_LOCALLY)

# --- End automatic project setup ---

import json
import platform
import subprocess
import sys
import uuid

import torch

print(platform.platform(), sys.version.split()[0], torch.__version__)

## Collect games and train

The sample deck contains 60 card IDs. Each observation belongs to the player making the current selection. The target for that player is +1 for a win, 0 for a draw, or −1 for a loss. Both players’ positions from a validation game stay together. After warmup, both seats use the same frozen weights and independent action decisions. Training then continues on the warmup and self-play games.

The game calls, AdamW learning rate of 0.0003 and value Huber loss with delta 0.2 follow the supplied example. This model has one value output. The host searches legal moves and uses the value of each resulting position to select option indices. [Kaggle training example](https://www.kaggle.com/code/kiyotah/reinforcement-learning-and-mcts-sample-code)

In [ ]:
# @title 2. Collect and train {display-mode: "form"}
WARMUP_GAMES = 4 # @param {type:"integer"}
SELF_PLAY_GAMES = 2 # @param {type:"integer"}
VALIDATION_GAMES = 2 # @param {type:"integer"}
EVALUATION_GAMES = 2 # @param {type:"integer"}
EPOCHS_PER_STAGE = 2 # @param {type:"integer"}
SEED = 7 # @param {type:"integer"}
OUTPUT_PARENT = "" # @param {type:"string"}
# @markdown Leave the output path blank to use the runtime's local storage. A mounted Drive folder will keep the files after the session ends.
parent = Path(OUTPUT_PARENT).expanduser().resolve() if OUTPUT_PARENT.strip() else ROOT / "project/results/deep-sets"
run_dir = parent / f"run-{uuid.uuid4().hex[:12]}"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Training on {device}. Saving to {run_dir}", flush=True)
subprocess.run([
    sys.executable, "-m", "project.train_deep_sets", "--output", str(run_dir), "--download",
    "--warmup-games", str(WARMUP_GAMES), "--self-play-games", str(SELF_PLAY_GAMES),
    "--validation-games", str(VALIDATION_GAMES), "--evaluation-games", str(EVALUATION_GAMES),
    "--epochs", str(EPOCHS_PER_STAGE), "--device", device, "--seed", str(SEED),
], cwd=ROOT, check=True)

## Read the result

Validation loss measures position-value error on separate games. The win/loss/draw tally comes from fresh games against the random agent with alternating seats. Two evaluation games can confirm that the checkpoint plays through the interface. A strength estimate needs many more games and opponents.

In [ ]:
report = json.loads((run_dir / "report.json").read_text())
for stage in ("warmup", "self_play", "validation", "evaluation"):
    data = report[stage]
    print(f"{stage:12} {len(data['games']):3} games  {data['positions']:5} decisions")
print("Best held-out Huber loss after self-play:", report["training"]["self_play"]["best_validation_loss"])
evaluation = report["evaluation"]
print("Against Random:", evaluation["wins"], "wins,", evaluation["losses"], "losses,", evaluation["draws"], "draws")
print("Self-play search counts:", report["self_play"]["agent_stats"])
print("Arena checkpoint:", run_dir / "model.pt")

## Verify the interface

Reload the trained weights, check saved decisions and compare full and incremental predictions. Select **Deep Sets checkpoint** in the team arena to use `model.pt` in another series. The checkpoint and report can be downloaded together below.

In [ ]:
# @title 3. Verify the run and optionally download the model {display-mode: "form"}
DOWNLOAD_MODEL = False # @param {type:"boolean"}
subprocess.run([sys.executable, "-m", "project.check_deep_sets_training", "--run-dir", str(run_dir)], cwd=ROOT, check=True)
if DOWNLOAD_MODEL:
    from zipfile import ZipFile, ZIP_DEFLATED
    archive = run_dir / "deep-sets-model.zip"
    with ZipFile(archive, "w", compression=ZIP_DEFLATED) as bundle:
        for name in ("model.pt", "report.json"):
            bundle.write(run_dir / name, arcname=name)
    if running_in_colab():
        from google.colab import files
        files.download(str(archive))
    else:
        print(archive)